In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, BertConfig, BertModel

c:\Users\13072\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#settings
seed=42 # set a constant seed, reach 91.820% in kaggle with 42 91.3 with seed 666.
train_fraction=1
validation_fraction=0
batch_size=128
Bert_Model  =  "google/bert_uncased_L-2_H-256_A-4"
tokenizer = AutoTokenizer.from_pretrained(Bert_Model, local_files_only=True)
tokenizer.model_max_length = 10**9
device = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = device == "cuda"
# we define negative and positive labels as 0 and 1 respectively, DONT CHANGE THIS
train_df=pd.read_csv("train.csv") # get the data
submission_data=pd.read_csv ("test.csv")
train_data = train_df[["review", "label"]]
train_data["label_id"] = train_data["label"].map({"negative": 0, "positive": 1}).astype(int) 
total_data=len(train_data)
rng = np.random.default_rng(seed)
indices = np.arange(total_data)
rng.shuffle(indices)
N_train = int(total_data * train_fraction)
N_val = int(total_data * validation_fraction)
# indices
train_indices = indices[:N_train] # get the indices
val_indices = indices[N_train:N_train + N_val]

# I dont know how to make this part parallel, maybe you can change this. but the speed is enough, I think...
# we can still optimize this using numpy vector indexing, but won't do much change to final speed
texts = train_data["review"].astype(str).tolist() #seperately create train
labels = train_data["label_id"].tolist()
train_texts = [texts[i] for i in train_indices]
val_texts = [texts[i] for i in val_indices]

train_labels = [labels[i] for i in train_indices]
#val_labels = [labels[i] for i in val_indices]




In [3]:

class ReviewCLSDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts) # we need texts and labels
        self.labels = labels
        self.max_length = 512 #512 tokens at max
        self.encodings = [self._encode_for_cls(text) for text in self.texts] # we use our own encoder

    def _encode_for_cls(self, text): # our encoding function
        token_ids = tokenizer.encode(str(text), add_special_tokens=True)
        # Because our bert only has 512 tokens, we need to truncate long reviews. 
        # for short reviews, we can just use the standard [CLS] review [SEP] format.
        if len(token_ids) <= self.max_length - 2:
            input_ids =token_ids

        else:
        # for long reviews, we keep the head and tail parts, and truncate the middle part.
        # we decide to use this method because the whole review is long
        # if you decide to just think about the head part the accuracy may not be high.
            head_len = self.max_length // 3 #1:2 is a reasonable choice for the sepeartion of head and tail and performs well
            tail_len = self.max_length - head_len
            head_ids = token_ids[:head_len]
            tail_ids = token_ids[-tail_len:]
            input_ids = (head_ids+tail_ids)
        
        token_type_ids = [0] * len(input_ids) # seperate sentence one from sentence two
        attention_mask = [1] * len(input_ids) # tell bert which tokens are real texts
        pad_len = self.max_length - len(input_ids) # make sure the final input length are of the same

        return { #return a dictionary
            "input_ids": input_ids + [tokenizer.pad_token_id] * pad_len,
            "attention_mask": attention_mask + [0] * pad_len,
            "token_type_ids": token_type_ids + [0] * pad_len,
        }

    def __len__(self):
        return len(self.texts) # return length

    def __getitem__(self, idx):
        encoding = self.encodings[idx]
        item = { # convert the lists to tensors
                "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
                "token_type_ids": torch.tensor(encoding["token_type_ids"], dtype=torch.long),
                "labels": torch.tensor(self.labels[idx], dtype=torch.long), }
        return item
    
train_dataset = ReviewCLSDataset(train_texts, train_labels) #instantiate the datasets
#val_dataset = ReviewCLSDataset(val_texts, val_labels)


In [4]:
#our model
class BertSmallCLSClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.2, num_labels=2): # 0.2 for dropout, 2 for binary classification
        super().__init__()

        self.config = BertConfig.from_pretrained( # load the configrations of BERT
            model_name,
            local_files_only=True, #only load from local
            num_labels=num_labels,
            id2label={0: "negative", 1: "positive"}, # convert number to text label
            label2id={"negative": 0, "positive": 1}, # cnovert text label to number
        )

        self.bert = BertModel.from_pretrained( # load the pretrained BERT model
            model_name,
            config=self.config,
            local_files_only=True,
        )
        for param in self.bert.parameters():
            param.requires_grad = True # we are fine-tuning the whole BERT rather than only the classifier
        self.classifier = nn.Sequential( # our own classifier
            nn.Linear(self.config.hidden_size, 128), # hidden size -> 128
            nn.ReLU(),
            nn.Dropout(dropout), # we need dropouts because we found that our model is overfitting
            nn.Linear(128, 64), # 128 -> 64
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32), # 64 -> 32
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_labels), # 32 -> num_labels
        )

    def forward(self, input_ids, attention_mask, token_type_ids): # forward function
        outputs = self.bert(input_ids=input_ids,attention_mask=attention_mask,token_type_ids=token_type_ids) # input tokenized review to BERT
        cls_output = outputs.last_hidden_state[:, 0, :] # we take the [CLS] token's output
        result = self.classifier(cls_output) # input [CLS] to our own classifier
        return result

model = BertSmallCLSClassifier(Bert_Model).to(device) # instantiate our model



Loading weights: 100%|██████████| 39/39 [00:00<00:00, 42956.37it/s]
[transformers] BertModel LOAD REPORT from: google/bert_uncased_L-2_H-256_A-4
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:

# we need adjust the batch size according to GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # instantiate dataloaders
criterion = nn.CrossEntropyLoss() # set the loss function

optimizer = torch.optim.AdamW( # we chose adamw to be our optimizer because it is more stable and more suitable for weight decay
    [
        {"params": model.bert.parameters(), "lr": 2e-5}, # we seperate the parameters of model into two groups and apply different learning rate. Small lr for BERT because it is already pretrained
        {"params": model.classifier.parameters(), "lr": 1e-4}, # bigger learning rate for our own classifier because it is randomly initialized
    ],
    weight_decay=0.005, # prevent parameters from growing too big, reduce overfitting
)



In [6]:

train_losses = [] # use lists to recorde the result of every epoch
train_accuracies = []

for epoch in range(9): # train for 9 rounds
    model.train() # switch the model to training mode
    total_train_loss = 0 #initialize
    train_correct = 0
    train_total = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device) # put the data in a batch to gpu or cpu, whichever device the model is on
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad() # clear the previous gradient
        pred = model( # forward propagation
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        loss = criterion(pred, labels) # calculate loss
        loss.backward() # backward propagation
        optimizer.step() # update parameters
        total_train_loss += loss.item() # accumulate loss
        predictions = torch.argmax(pred, dim=1) # take the group of highest score as prediction result
        train_correct += (predictions == labels).sum().item() # count the correct predictions
        train_total += labels.size(0) # count the total number
    avg_train_loss = total_train_loss / len(train_loader) #calculate average loss
    train_acc = train_correct / train_total # calculate accuracy
    train_losses.append(avg_train_loss)
    train_accuracies.append(train_acc)
    print("epoch:", epoch + 1)
    print("train loss:", avg_train_loss)
    print("train accuracy:", train_acc)
    print()

# Generate submission.csv
test_texts = submission_data["review"].astype(str).tolist() #create required component for test
test_dataset = ReviewCLSDataset(test_texts, [0] * len(test_texts))
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

test_predictions = []
model.eval() # switch to evaluation mode
with torch.no_grad(): # don't need to calculate gradient
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device) # put the data in a batch to gpu or cpu, whichever device the model is on
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)

        pred = model( # forward propagation
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        test_predictions.extend(torch.argmax(pred, dim=1).cpu().tolist())

id_column = "Id"
label_map = {0: "negative", 1: "positive"} # conver 0 -> negative, 1 -> positive
submission = pd.DataFrame({ # a new dataframe with two colomns, id and label
    "Id": submission_data[id_column], # get the ids
    "label": [label_map[p] for p in test_predictions], # convert the prediction result from number to text
})
submission.to_csv("submission.csv", index=False) # write to csv
submission.head()

epoch: 1
train loss: 0.48145216948945413
train accuracy: 0.7669555555555555

epoch: 2
train loss: 0.31436639177528297
train accuracy: 0.8727777777777778

epoch: 3
train loss: 0.2725351049818776
train accuracy: 0.8905333333333333

epoch: 4
train loss: 0.24404259110716256
train accuracy: 0.9028888888888889

epoch: 5
train loss: 0.2194687113250521
train accuracy: 0.9145777777777778

epoch: 6
train loss: 0.1949574367608875
train accuracy: 0.9247555555555556

epoch: 7
train loss: 0.17758826147341591
train accuracy: 0.932

epoch: 8
train loss: 0.15995667490642518
train accuracy: 0.9396222222222222

epoch: 9
train loss: 0.14519897126592696
train accuracy: 0.9459111111111111



,Id,label
0,33554,positive
1,9428,positive
2,200,negative
3,12448,positive
4,39490,negative
